## Problem Statement
Twitter has become an important communication channel in times of emergency. The ubiquitousness of smartphones enables people to announce an emergency they’re observing in real-time. Because of this, more agencies are interested in programatically monitoring Twitter (i.e. disaster relief organizations and news agencies).

In this competition, we’re challenged to build a machine learning model that predicts which Tweets are about real disasters and which one’s aren’t. We have access to a dataset of 10,000 tweets that were hand classified.

## Import libraries

In [1]:
!pip install tweet-preprocessor

  Created wheel for tweet-preprocessor: filename=tweet_preprocessor-0.5.0-cp36-none-any.whl size=7947 sha256=4a3b6f6f67179d92ad48b90f8db584aac635e6124a7c2b64a5e1f3ae83629a48
  Stored in directory: /tmp/.cache/pip/wheels/1b/27/cc/49938e98a2470802ebdefae9d2b3f524768e970c1ebbe2dc4a
Successfully built tweet-preprocessor


In [2]:
import os
from google.cloud import storage, automl_v1beta1 as automl
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn import feature_extraction, linear_model, model_selection, preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import scipy as sp

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, KFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

from sklearn.naive_bayes import GaussianNB, MultinomialNB

import seaborn as sns
import matplotlib.pyplot as plt 

from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
from nltk.stem import WordNetLemmatizer
import nltk
from collections import Counter
import string
import re
from nltk.corpus import wordnet as wn
from statistics import mean 
import preprocessor 

In [3]:
from sklearn import metrics

## Common libraries

This section of code cleans the tweet text using tweet preprocesser library

In [4]:
stop = set(STOPWORDS).union(set(['FAV' , 'RT']))
lemma = WordNetLemmatizer()
preprocessor.set_options(preprocessor.OPT.URL, preprocessor.OPT.MENTION, preprocessor.OPT.NUMBER, preprocessor.OPT.RESERVED)

def clean(text):   
    text = preprocessor.clean(text)
    text = re.sub(r'[^\w\s]','',text)
    stop_free = " ".join([i for i in text.split(' ') if (i not in stop)])
    normalized = " ".join(lemma.lemmatize(word) for word in stop_free.split())
    return normalized

## Load Data

In [5]:
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

## Clean data

In [6]:
train_df.text = train_df.text.apply(clean)
test_df.text = test_df.text.apply(clean)

In [7]:
train_df.head(5)

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds Reason earthquake May ALLAH Forgive u,1
1,4,NaN,NaN,Forest fire near La Ronge Sask Canada,1
2,5,NaN,NaN,All resident asked shelter place notified offi...,1
3,6,NaN,NaN,000 people receive wildfire evacuation order C...,1
4,7,NaN,NaN,Just got sent photo Ruby Alaska smoke wildfire...,1


In [8]:
test_df.head(5)

,id,keyword,location,text
0,0,NaN,NaN,Just happened terrible car crash
1,2,NaN,NaN,Heard earthquake different city stay safe ever...
2,3,NaN,NaN,forest fire spot pond goose fleeing across str...
3,9,NaN,NaN,Apocalypse lighting Spokane wildfire
4,11,NaN,NaN,Typhoon Soudelor kill China Taiwan


## AutoML Model

In [9]:
# Set your own project id here
PROJECT_ID = 'automl-kaggle-263107'

In [10]:
#REPLACE THIS WITH A NEW BUCKET NAME. NOTE: BUCKET NAMES MUST BE GLOBALLY UNIQUE
BUCKET_NAME = 'automl-disaster-tweet-cleaned'
#Note: the bucket_region must be us-central1.
BUCKET_REGION = 'us-central1'

In [11]:
storage_client = storage.Client(project=PROJECT_ID)
tables_gcs_client = automl.GcsClient(client=storage_client, bucket_name=BUCKET_NAME)
automl_client = automl.AutoMlClient()
# Note: AutoML Tables currently is only eligible for region us-central1. 
prediction_client = automl.PredictionServiceClient()
# Note: This line runs unsuccessfully without each one of these parameters
tables_client = automl.TablesClient(project=PROJECT_ID, region=BUCKET_REGION, client=automl_client, gcs_client=tables_gcs_client, prediction_client=prediction_client)

In [12]:
# Create your GCS Bucket with your specified name and region (if it doesn't already exist)
bucket = storage.Bucket(storage_client, name=BUCKET_NAME)
if not bucket.exists():
    bucket.create(location=BUCKET_REGION)

In [13]:
def upload_blob(bucket_name, source_file_name, destination_blob_name):
    """Uploads a file to the bucket. https://cloud.google.com/storage/docs/ """
    bucket = storage_client.get_bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print('File {} uploaded to {}.'.format(
        source_file_name,
        destination_blob_name))
    
def download_to_kaggle(bucket_name,destination_directory,file_name,prefix=None):
    """Takes the data from your GCS Bucket and puts it into the working directory of your Kaggle notebook"""
    os.makedirs(destination_directory, exist_ok = True)
    full_file_path = os.path.join(destination_directory, file_name)
    blobs = storage_client.list_blobs(bucket_name,prefix=prefix)
    for blob in blobs:
        blob.download_to_filename(full_file_path)

In [14]:
test_df.head(5)

,id,keyword,location,text
0,0,NaN,NaN,Just happened terrible car crash
1,2,NaN,NaN,Heard earthquake different city stay safe ever...
2,3,NaN,NaN,forest fire spot pond goose fleeing across str...
3,9,NaN,NaN,Apocalypse lighting Spokane wildfire
4,11,NaN,NaN,Typhoon Soudelor kill China Taiwan


In [15]:
# Select the text body and the target value, for sending to AutoML
train_df[['id','text','target']].to_csv('/kaggle/working/train.csv', index=False) 
test_df[['id','text']].to_csv('/kaggle/working/test.csv', index=False) 

In [16]:
upload_blob(BUCKET_NAME, '/kaggle/working/train.csv', 'train.csv')
upload_blob(BUCKET_NAME, '/kaggle/working/test.csv', 'test.csv')

File /kaggle/working/train.csv uploaded to train.csv.
File /kaggle/working/test.csv uploaded to test.csv.


In [17]:
dataset_display_name = 'tweet_disaster_cleaned'
new_dataset = False
try:
    dataset = tables_client.get_dataset(dataset_display_name=dataset_display_name)
except:
    new_dataset = True
    dataset = tables_client.create_dataset(dataset_display_name)

In [18]:
# gcs_input_uris have the familiar path of gs://BUCKETNAME//file

if new_dataset:
    gcs_input_uris = ['gs://' + BUCKET_NAME + '/train.csv']

    import_data_operation = tables_client.import_data(
        dataset=dataset,
        gcs_input_uris=gcs_input_uris
    )
    print('Dataset import operation: {}'.format(import_data_operation))

    # Synchronous check of operation status. Wait until import is done.
    import_data_operation.result()

In [19]:
print(dataset)

name: "projects/308167682590/locations/us-central1/datasets/TBL2730802054325862400"
display_name: "tweet_disaster_cleaned"
create_time {
  seconds: 1577773329
  nanos: 667270000
}
etag: "AB3BwFpNOkO6b6LJ9iWHRipCn98EvihRTwJ0KxRPnrSSDxt-EPoB2MMNIAFChRUMQNE="
example_count: 7613
tables_dataset_metadata {
  primary_table_spec_id: "3181203798504767488"
  target_column_spec_id: "6853082253090619392"
  target_column_correlations {
    key: "9158925262304313344"
    value {
      cramers_v: 0.14138388654003808
    }
  }
  stats_update_time {
    seconds: 1577780990
    nanos: 114000000
  }
}



In [20]:
ID_COLUMN = 'id'

In [21]:
TARGET_COLUMN = 'target'

tables_client.set_target_column(
    dataset=dataset,
    column_spec_display_name=TARGET_COLUMN
)

name: "projects/308167682590/locations/us-central1/datasets/TBL2730802054325862400"
display_name: "tweet_disaster_cleaned"
create_time {
  seconds: 1577773329
  nanos: 667270000
}
etag: "AB3BwFpz24qFSP3WHsL1qBkPmqSOFW_O8wVYVPULCt8Do_zsXdNkOkjydJW5V5UWvLou"
example_count: 7613
tables_dataset_metadata {
  primary_table_spec_id: "3181203798504767488"
  target_column_spec_id: "6853082253090619392"
  target_column_correlations {
    key: "9158925262304313344"
    value {
      cramers_v: 0.14138388654003808
    }
  }
  stats_update_time {
    seconds: 1577780990
    nanos: 114000000
  }
}

In [22]:
# Make all columns nullable (except the Target and ID Column)
for col in tables_client.list_column_specs(PROJECT_ID,BUCKET_REGION,dataset.name):
    if TARGET_COLUMN in col.display_name or ID_COLUMN in col.display_name:
        continue
    tables_client.update_column_spec(PROJECT_ID,
                                     BUCKET_REGION,
                                     dataset.name,
                                     column_spec_display_name=col.display_name,
                                     type_code=col.data_type.type_code,
                                     nullable=True)

In [23]:
# Train the model. This will take hours (up to your budget). AutoML will early stop if it finds an optimal solution before your budget.
# On this dataset, AutoML usually stops around 2000 milli-hours (2 hours)

TRAIN_BUDGET = 1000 # (specified in milli-hours, from 1000-72000)
model = None
model_display_name = 'tweet_disaster_model_clean'
try:
    model = tables_client.get_model(model_display_name=model_display_name)
except:
    response = tables_client.create_model(
        model_display_name,
        dataset=dataset,
        train_budget_milli_node_hours=TRAIN_BUDGET,
        exclude_column_spec_names=[TARGET_COLUMN,ID_COLUMN]
    )
    print('Create model operation: {}'.format(response.operation))
    # Wait until model training is done.
    model = response.result()
print(model)

name: "projects/308167682590/locations/us-central1/models/TBL2399446633109520384"
display_name: "tweet_disaster_model_clean"
dataset_id: "TBL2730802054325862400"
create_time {
  seconds: 1577773698
  nanos: 775356000
}
deployment_state: UNDEPLOYED
update_time {
  seconds: 1577776850
  nanos: 351671000
}
tables_model_metadata {
  target_column_spec {
    name: "projects/308167682590/locations/us-central1/datasets/TBL2730802054325862400/tableSpecs/3181203798504767488/columnSpecs/6853082253090619392"
    data_type {
      type_code: CATEGORY
    }
    display_name: "target"
  }
  input_feature_column_specs {
    name: "projects/308167682590/locations/us-central1/datasets/TBL2730802054325862400/tableSpecs/3181203798504767488/columnSpecs/4547239243876925440"
    data_type {
      type_code: STRING
      nullable: true
    }
    display_name: "text"
  }
  optimization_objective: "MAXIMIZE_AU_ROC"
  tables_model_column_info {
    column_spec_name: "projects/308167682590/locations/us-central1/

In [24]:
gcs_input_uris = 'gs://' + BUCKET_NAME + '/test.csv'
gcs_output_uri_prefix = 'gs://' + BUCKET_NAME + '/predictions'

batch_predict_response = tables_client.batch_predict(
    model=model, 
    gcs_input_uris=gcs_input_uris,
    gcs_output_uri_prefix=gcs_output_uri_prefix,
)
print('Batch prediction operation: {}'.format(batch_predict_response.operation))
# Wait until batch prediction is done.
batch_predict_result = batch_predict_response.result()
batch_predict_response.metadata

Batch prediction operation: name: "projects/308167682590/locations/us-central1/operations/TBL6109362692458283008"
metadata {
  type_url: "type.googleapis.com/google.cloud.automl.v1beta1.OperationMetadata"
  value: "\032\013\010\355\267\273\360\005\020\370\264\241L\"\013\010\355\267\273\360\005\020\370\264\241L\202\0011\n/\n-\n+gs://automl-disaster-tweet-cleaned/test.csv"
}



create_time {
  seconds: 1578032109
  nanos: 159931000
}
update_time {
  seconds: 1578032240
  nanos: 287130000
}
batch_predict_details {
  input_config {
    gcs_source {
      input_uris: "gs://automl-disaster-tweet-cleaned/test.csv"
    }
  }
  output_info {
    gcs_output_directory: "gs://automl-disaster-tweet-cleaned/predictions/prediction-tweet_disaster_model_clean-2020-01-03T06:15:09.011Z"
  }
}

In [25]:
# The output directory for the prediction results exists under the response metadata for the batch_predict operation
# Specifically, under metadata --> batch_predict_details --> output_info --> gcs_output_directory
# Then, you can remove the first part of the output path that contains the GCS Bucket information to get your desired directory
gcs_output_folder = batch_predict_response.metadata.batch_predict_details.output_info.gcs_output_directory.replace('gs://' + BUCKET_NAME + '/','')
download_to_kaggle(BUCKET_NAME,'/kaggle/working','submissions.csv', prefix=gcs_output_folder)

In [26]:
preds_df = pd.read_csv("/kaggle/working/submissions.csv")
preds_df = preds_df.sort_values(by=['id'])
preds_df['target'] = (preds_df['target_1_score'] >= 0.5).astype(int)

In [27]:
preds_df.head(50)

,id,text,target_0_score,target_1_score,target
3053,0,Just happened terrible car crash,0.306924,0.693076,1
1311,2,Heard earthquake different city stay safe ever...,0.343647,0.656353,1
704,3,forest fire spot pond goose fleeing across str...,0.393055,0.606945,1
1472,9,Apocalypse lighting Spokane wildfire,0.378672,0.621328,1
2670,11,Typhoon Soudelor kill China Taiwan,0.151694,0.848306,1
1534,12,Were shakingIts earthquake,0.379708,0.620292,1
2128,21,Theyd probably still show life Arsenal yesterd...,0.813189,0.186811,0
2715,22,Hey How,0.803801,0.196199,0
1098,27,What nice hat,0.800588,0.199412,0
780,29,Fuck,0.758302,0.241698,0


In [28]:
preds_df[['id','target']].to_csv("submission.csv", index=False)

## Simple Classifier Models

### Building vectors

The theory behind the model we'll build in this notebook is pretty simple: the words contained in each tweet are a good indicator of whether they're about a real disaster or not (this is not entirely correct, but it's a great place to start).

We'll use scikit-learn's `CountVectorizer` to count the words in each tweet and turn them into data our machine learning model can process.

Note: a `vector` is, in this context, a set of numbers that a machine learning model can work with. We'll look at one in just a second.

In [29]:
tfidf_vectorizer = feature_extraction.text.TfidfVectorizer(ngram_range = (1,2), stop_words='english',strip_accents='unicode')

In [30]:
train_vectors = tfidf_vectorizer.fit_transform(train_df["text"])

## note that we're NOT using .fit_transform() here. Using just .transform() makes sure
# that the tokens in the train vectors are the only ones mapped to the test vectors - 
# i.e. that the train and test vectors use the same set of tokens.
test_vectors = tfidf_vectorizer.transform(test_df["text"])

In [31]:
train_vectors.todense().shape

(7613, 55129)

### Random Forest classifier for non-text data

In [32]:
# define X and y
feature_cols = ['keyword', 'location']
X = train_df[feature_cols]
y = train_df.target
one_hot_encoded_training_predictors = pd.get_dummies(X)
clf = RandomForestClassifier(n_estimators = 100)
scores = model_selection.cross_val_score(clf, one_hot_encoded_training_predictors, y, cv=5, scoring="f1")
scores

array([0.47058824, 0.43953733, 0.54298203, 0.52428147, 0.20306513])

In [33]:
clf.fit(one_hot_encoded_training_predictors, y)

RandomForestClassifier(bootstrap=True, class_weight=None, criterion='gini',
                       max_depth=None, max_features='auto', max_leaf_nodes=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=100,
                       n_jobs=None, oob_score=False, random_state=None,
                       verbose=0, warm_start=False)

### Ridge and SVM classifier for text data

As we mentioned above, we think the words contained in each tweet are a good indicator of whether they're about a real disaster or not. The presence of particular word (or set of words) in a tweet might link directly to whether or not that tweet is real.

What we're assuming here is a _linear_ connection. So let's build a linear model and see!

In [34]:
## Our vectors are really big, so we want to push our model's weights
## toward 0 without completely discounting different words - ridge regression 
## is a good way to do this.
clf = linear_model.RidgeClassifier()
# clf = SVC(C=1.0, cache_size=200, class_weight=None, coef0=0.0,
#     decision_function_shape='ovr', degree=3, gamma=0.7, kernel='rbf',
#     max_iter=-1, probability=False, random_state=None, shrinking=True,
#     tol=0.001, verbose=False)
#clf = linear_model.LogisticRegression() #same as ridge
#clf = DecisionTreeClassifier() #bad performance
#clf=RandomForestClassifier(n_estimators = 100) #bad performance

In [35]:
# Let's test our model and see how well it does on the training data. For this we'll use `cross-validation` - where we train on a portion of the known data, then validate it with the rest. If we do this several times (with different portions) we can get a good idea for how a particular model or method performs.

# The metric for this competition is F1, so let's use that here.
scores = model_selection.cross_val_score(clf, train_vectors, train_df["target"], cv=10, scoring="f1")
scores

array([0.61689587, 0.4137931 , 0.4911032 , 0.44606947, 0.49819495,
       0.4853229 , 0.54511278, 0.41949153, 0.68412439, 0.70376432])

In [36]:
clf.fit(train_vectors, train_df["target"])

RidgeClassifier(alpha=1.0, class_weight=None, copy_X=True, fit_intercept=True,
                max_iter=None, normalize=False, random_state=None,
                solver='auto', tol=0.001)

#### SVM

In [37]:
parameters = { 
    'gamma': [0.7, 1, 'auto', 'scale']
}
clf = GridSearchCV(SVC(kernel='rbf'), parameters, cv=5, n_jobs=-1, scoring="f1").fit(train_vectors, train_df["target"]) #SVM-slightly better than ridge

In [38]:
clf.best_estimator_

SVC(C=1.0, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma=0.7, kernel='rbf',
    max_iter=-1, probability=False, random_state=None, shrinking=True,
    tol=0.001, verbose=False)

In [39]:
clf.best_score_

0.4112342021682781

Let's do predictions on our training set and build a submission for the competition.

In [40]:
sample_submission = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
sample_submission["target"] = clf.predict(test_vectors)
df = pd.DataFrame({'text' : test_df['text'], 'prediction' : sample_submission["target"]})

In [41]:
sample_submission.to_csv("submission1.csv", index=False)

## Tensorflow model

In [42]:
import tensorflow as tf
print(tf.__version__)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

2.1.0-rc0


In [43]:
X = train_df["text"]
y = train_df["target"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

In [44]:
vocab_size = 10000
embedding_dim = 16
max_length = 30 #Based on data exploration
trunc_type='post'
padding_type='post'
oov_tok = "<OOV>"

In [45]:
tokenizer = Tokenizer(num_words = vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(X_train)
padded = pad_sequences(sequences,maxlen=max_length, padding=padding_type, truncating=trunc_type)

testing_sequences = tokenizer.texts_to_sequences(X_val)
testing_padded = pad_sequences(testing_sequences,maxlen=max_length, padding=padding_type, truncating=trunc_type)

In [46]:
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

def decode_sentence(text):
    return ' '.join([reverse_word_index.get(i, '?') for i in text])

print(decode_sentence(padded[0]))

ashes australiaûªs collapse trent bridge among worst history england bundled australia ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?


In [47]:
print(X_train.values[0])

Ashes AustraliaÛªs collapse Trent Bridge among worst history England bundled Australia


In [48]:
len(word_index)

11997

#### Embedding model

In [49]:
# Note this is the 100 dimension version of GloVe from Stanford
# I unzipped and hosted it on my site to make this notebook easier
!wget --no-check-certificate \
    https://storage.googleapis.com/laurencemoroney-blog.appspot.com/glove.6B.100d.txt \
    -O /tmp/glove.6B.100d.txt
embeddings_index = {};
vocab_size=len(word_index)
embedding_dim = 100
with open('/tmp/glove.6B.100d.txt') as f:
    for line in f:
        values = line.split();
        word = values[0];
        coefs = np.asarray(values[1:], dtype='float32');
        embeddings_index[word] = coefs;

embeddings_matrix = np.zeros((vocab_size+1, embedding_dim));
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word);
    if embedding_vector is not None:
        embeddings_matrix[i] = embedding_vector;

--2020-01-03 06:21:21--  https://storage.googleapis.com/laurencemoroney-blog.appspot.com/glove.6B.100d.txt
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.132.128, 2607:f8b0:4001:c14::80
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.132.128|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 347116733 (331M) [text/plain]
Saving to: ‘/tmp/glove.6B.100d.txt’

/tmp/glove.6B.100d. 100%[===================>] 331.04M  62.3MB/s    in 6.6s    

2020-01-03 06:21:28 (50.2 MB/s) - ‘/tmp/glove.6B.100d.txt’ saved [347116733/347116733]



In [50]:
model = tf.keras.Sequential([
    #Embedding
#     tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length), #Each sentence will have 120 words
#     tf.keras.layers.Flatten(),
#     tf.keras.layers.Dense(6, activation='relu'),
#     tf.keras.layers.Dense(1, activation='sigmoid')
    
    #Word embedding with pooling
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
    
    #LSTM
#     tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
#     tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
#     tf.keras.layers.Dense(24, activation='relu'),
#     tf.keras.layers.Dense(1, activation='sigmoid')
    
    #Multi Layer LSTM - Best performing
#     tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
#     tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
#     tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
#     tf.keras.layers.Dense(64, activation='relu'),
#     tf.keras.layers.Dense(1, activation='sigmoid')
    
    #Glove embedding, Drop out etc
#     tf.keras.layers.Embedding(vocab_size+1, embedding_dim, input_length=max_length, weights=[embeddings_matrix], trainable=False),
#     tf.keras.layers.Dropout(0.2),
#     tf.keras.layers.Conv1D(64, 5, activation='relu'),
#     tf.keras.layers.MaxPooling1D(pool_size=4),
#     tf.keras.layers.LSTM(64),
#     tf.keras.layers.Dense(1, activation='sigmoid')
    
    #GRU
#     tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
#     tf.keras.layers.Bidirectional(tf.keras.layers.GRU(32)),
#     tf.keras.layers.Dense(6, activation='relu'),
#     tf.keras.layers.Dense(1, activation='sigmoid')
    
    #ConvD
#     tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
#     tf.keras.layers.Conv1D(128, 5, activation='relu'),
#     tf.keras.layers.GlobalAveragePooling1D(),
#     tf.keras.layers.Dense(6, activation='relu'),
#     tf.keras.layers.Dense(1, activation='sigmoid') 
])
#model.compile(optimizer='adam', loss=f1_loss, metrics=['accuracy', f1])
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy',tf.keras.metrics.AUC()])
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding (Embedding)        (None, 30, 100)           1199700   
_________________________________________________________________
global_average_pooling1d (Gl (None, 100)               0         
_________________________________________________________________
dense (Dense)                (None, 24)                2424      
_________________________________________________________________
dense_1 (Dense)              (None, 1)                 25        
Total params: 1,202,149
Trainable params: 1,202,149
Non-trainable params: 0
_________________________________________________________________


In [51]:
num_epochs = 3
history = model.fit(padded, y_train, epochs=num_epochs, validation_data=(testing_padded, y_val))

Train on 5329 samples, validate on 2284 samples
Epoch 1/3
5329/5329 [==============================] - 5s 958us/sample - loss: 0.6687 - accuracy: 0.5733 - auc: 0.6587 - val_loss: 0.6292 - val_accuracy: 0.6484 - val_auc: 0.8450
Epoch 2/3
5329/5329 [==============================] - 3s 527us/sample - loss: 0.5177 - accuracy: 0.7705 - auc: 0.8799 - val_loss: 0.4840 - val_accuracy: 0.7833 - val_auc: 0.8606
Epoch 3/3
5329/5329 [==============================] - 3s 526us/sample - loss: 0.3320 - accuracy: 0.8720 - auc: 0.9384 - val_loss: 0.4424 - val_accuracy: 0.8087 - val_auc: 0.8643


In [52]:
model_loss = pd.DataFrame(model.history.history)
model_loss.head()

,loss,accuracy,auc,val_loss,val_accuracy,val_auc
0,0.668702,0.573278,0.658673,0.629177,0.648424,0.845026
1,0.517701,0.770501,0.879900,0.483963,0.783275,0.860619
2,0.331957,0.872021,0.938384,0.442402,0.808669,0.864284


In [53]:
#model_loss[['accuracy','val_accuracy']].plot(ylim=[0,1]);
model_loss[['auc_9','val_auc_9']].plot(ylim=[0,1]);

KeyError: "None of [Index(['auc_9', 'val_auc_9'], dtype='object')] are in the [columns]"

In [54]:
testing_sequences2 = tokenizer.texts_to_sequences(test_df.text)
testing_padded2 = pad_sequences(testing_sequences2, maxlen=max_length, padding=padding_type, truncating=trunc_type)

In [55]:
probabilities = model.predict(testing_padded2)

In [56]:
predictions = (probabilities > 0.5).astype(int)
predictions = np.ndarray.flatten(predictions)
pd.value_counts(predictions)

0    1992
1    1271
dtype: int64

In [57]:
original_test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
df = pd.DataFrame({'text' : original_test_df['text'],'cleaned_text' : test_df['text'], 'prediction' : predictions,'probabilities' : np.ndarray.flatten(probabilities)})
df.to_csv("test_df.csv", index=False)

In [58]:
df.values[50:100]

array([["Stop saying 'I Wish' and start saying 'I Will'. \x89ÛÒ Unknown",
        'Stop saying I Wish start saying I Will ÛÒ Unknown', 0,
        0.07598698139190674],
       ["I want to go to Aftershock in October because it has all the bands I listen to and #NXT! Can't afford it yet though. #gradschoolapps",
        'I want go Aftershock October band I listen NXT Cant afford yet though gradschoolapps',
        0, 0.016247481107711792],
       ["'We are still living in the aftershock of Hiroshima people are still the scars of history.' - Edward Bond http://t.co/engTl5wrGp",
        'We still living aftershock Hiroshima people still scar history Edward Bond',
        1, 0.7309558391571045],
       ['320 [IR] ICEMOON [AFTERSHOCK] | http://t.co/THyzOMVWU0 | @djicemoon | #Dubstep #TrapMusic #DnB #EDM #Dance #Ices\x89Û_ http://t.co/83jOO0xk29',
        'IR ICEMOON AFTERSHOCK Dubstep TrapMusic DnB EDM Dance IcesÛ_',
        0, 0.006374537944793701],
       ['Aftershock https://t.co/Ecy4U623

In [59]:
sample_submission = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
sample_submission["target"] = predictions
sample_submission.to_csv("submission.csv", index=False)

## Tensorflow Hub  - Universal Sentence Encoder + LightGBM

In [60]:
import tensorflow_hub as hub
import lightgbm as lgb
from lightgbm import LGBMClassifier

In [61]:
module_url = "https://tfhub.dev/google/nnlm-en-dim128/2"
embed = hub.KerasLayer(module_url)
embeddings = embed(["A long sentence.", "single-word",
                  "http://example.com"])
print(embeddings.shape)  #(3,128)

(3, 128)


In [62]:
embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/3")

In [63]:
X_train_embeddings = embed(train_df.text.values)
X_test_embeddings = embed(test_df.text.values)

In [64]:
params = {
    'learning_rate': 0.04,
    'n_estimators': 1500,
    'colsample_bytree': 0.4,
    'metric':'auc'
}

In [65]:
text_clf = LGBMClassifier(**params)

In [66]:
text_clf.fit(X_train_embeddings['outputs'][:5000,:], train_df.target.values[:5000],
             eval_set=[(X_train_embeddings['outputs'][:5000,:], train_df.target.values[:5000]),
                       (X_train_embeddings['outputs'][5000:,:], train_df.target.values[5000:])],
             verbose=200, early_stopping_rounds=20,
            )


Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[126]	valid_0's auc: 0.991559	valid_1's auc: 0.874544


LGBMClassifier(boosting_type='gbdt', class_weight=None, colsample_bytree=0.4,
               importance_type='split', learning_rate=0.04, max_depth=-1,
               metric='auc', min_child_samples=20, min_child_weight=0.001,
               min_split_gain=0.0, n_estimators=1500, n_jobs=-1, num_leaves=31,
               objective=None, random_state=None, reg_alpha=0.0, reg_lambda=0.0,
               silent=True, subsample=1.0, subsample_for_bin=200000,
               subsample_freq=0)

In [67]:
text_clf.fit(X_train_embeddings['outputs'][:5000,:], train_df.target.values[:5000])
Y_pred = text_clf.predict(X_train_embeddings['outputs'][5000:])

In [68]:
print(metrics.classification_report(train_df.target[5000:], Y_pred, digits=3),) 
print(metrics.confusion_matrix(train_df.target[5000:], Y_pred))

              precision    recall  f1-score   support

           0      0.776     0.873     0.822      1436
           1      0.817     0.693     0.750      1177

    accuracy                          0.792      2613
   macro avg      0.797     0.783     0.786      2613
weighted avg      0.795     0.792     0.789      2613

[[1253  183]
 [ 361  816]]


In [69]:
text_clf.fit(X_train_embeddings['outputs'], train_df.target.values)
pred_test = text_clf.predict(X_test_embeddings['outputs'])

In [70]:
df = pd.DataFrame({'cleaned_text' : test_df['text'], 'prediction' : pred_test})
df.head(20)

,cleaned_text,prediction
0,Just happened terrible car crash,1
1,Heard earthquake different city stay safe ever...,1
2,forest fire spot pond goose fleeing across str...,1
3,Apocalypse lighting Spokane wildfire,1
4,Typhoon Soudelor kill China Taiwan,1
5,Were shakingIts earthquake,1
6,Theyd probably still show life Arsenal yesterd...,0
7,Hey How,0
8,What nice hat,0
9,Fuck,0


In [71]:
sample_submission = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
sample_submission["target"] = pred_test
sample_submission.to_csv("submission.csv", index=False)